#Author: Mike Bollinger
    # BOEM - Carolina Long Bay
    # May 31, 2024
    # Feed in CSV from Sample point generation script, Create a feature service and update with sample sites
    ######################################################################

# UPDATED v1 06.07.2024
#This is the first version - Mike Bollinger

In [69]:
from arcgis.gis import GIS
from arcgis.features import FeatureLayerCollection
import pandas as pd


# Connect to ArcGIS Online
gis = GIS("home")
#Existing Feature Service ID
ExistingFS = 'b92b5ed439cb48ae81b051271f83d893' #If you don't have one of these then run the code, find the FS then fill in the ID
####Path to new data####
new_data_dbf = pd.read_csv(r"C:\Users\Mike.Bollinger\Documents\R\NF-24-06_TEST\Outputs\SamplingPoints\P1_StratSamp.csv")
new_data_dbf.shape

###BOEM
new_columns = ['AUV_Completed_Date', 'ROV_Completed_Date','DropCam_Completed_Date','Status']
for col in new_columns:
    new_data_dbf[col] = 'Not Complete'
new_data_dbf.head()

,SiteID,Lon,Lat,Easting,Northing,SurveyPrio,Design,Strata,inclusion.probabilities,DateGenerated,DayNumber,AUVCompletedDate,ROVCompletedDate,DropCamCompletedDate,Status
0,47,-77.879214,33.493515,789953.5,3710360.5,1,Stratified,1,2.240000e-08,6/7/2024,DN24159,Not Complete,Not Complete,Not Complete,Not Complete
1,48,-77.874654,33.517982,790295.5,3713087.5,1,Stratified,2,2.310000e-07,6/7/2024,DN24159,Not Complete,Not Complete,Not Complete,Not Complete
2,49,-77.861624,33.470534,791665.5,3707860.5,1,Stratified,1,2.240000e-08,6/7/2024,DN24159,Not Complete,Not Complete,Not Complete,Not Complete
3,50,-77.869457,33.483019,790895.5,3709223.5,1,Stratified,2,2.310000e-07,6/7/2024,DN24159,Not Complete,Not Complete,Not Complete,Not Complete
4,51,-77.864080,33.478791,791409.5,3708769.5,1,Stratified,2,2.310000e-07,6/7/2024,DN24159,Not Complete,Not Complete,Not Complete,Not Complete


In [70]:
####Path to existing data####
JoinedCSVpath = r"C:\Users\Mike.Bollinger\Documents\R\NF-24-06_TEST\Outputs\SamplingPoints\Joined_Sampling_Points.csv"

Run #Create Feature service block if this is the first iteration of data you're adding to AGOL

In [71]:
####Create Feature service
###Path to first file to create the FS
firstCSVpath = r"C:\Users\Mike.Bollinger\Documents\R\NF-24-06_TEST\Outputs\SamplingPoints\RegSample.csv"
try:
    ssItem = gis.content.get(ExistingFS)
    print("Feature Service already created, Loaded it")
except Exception:
    print("No Feature Service found, creating one...")
    firstCSV = pd.read_csv(firstCSVpath)
    ###BOEM
    new_columns = ['AUVCompletedDate', 'ROVCompletedDate','DropCamCompletedDate','Status']
    for col in new_columns:
        firstCSV[col] = 'Not Completed'


    firstCSV.to_csv(JoinedCSVpath)
    #set properties
    item_prop = {'title':'BOEM CLB Sample Sites'}
    csv_item = gis.content.add(item_properties = item_prop, data = firstCSVpath)
    csv_item
    #Publish Feature layer
    ssItem = csv_item.publish()
    ssItem
    print("Feature Service created")
print(ssItem)

No Feature Service found, creating one...
Feature Service created
<Item title:"BOEM CLB Sample Sites" type:Feature Layer Collection owner:mike.bollinger_noaa>


In [75]:
    firstCSV.head()
    #firstCSV.shape

,SiteID,Lon,Lat,Easting,Northing,SurveyPrio,Design,DateGenerated,DayNumber,AUV_Completed_Date,ROV_Completed_Date,DropCam_Completed_Date,Status,AUVCompletedDate,ROVCompletedDate,DropCamCompletedDate
0,1,-77.854471,33.402887,792557.7910,3700376.285,3,Regular,6/7/2024,DN24159,Not Completed,Not Completed,Not Completed,Not Completed,Not Completed,Not Completed,Not Completed
1,2,-77.839455,33.402505,793956.2123,3700376.285,3,Regular,6/7/2024,DN24159,Not Completed,Not Completed,Not Completed,Not Completed,Not Completed,Not Completed,Not Completed
2,3,-77.824440,33.402122,795354.6337,3700376.285,6,Regular,6/7/2024,DN24159,Not Completed,Not Completed,Not Completed,Not Completed,Not Completed,Not Completed,Not Completed
3,4,-77.809425,33.401736,796753.0551,3700376.285,6,Regular,6/7/2024,DN24159,Not Completed,Not Completed,Not Completed,Not Completed,Not Completed,Not Completed,Not Completed
4,5,-77.884053,33.416241,789760.9482,3701774.707,4,Regular,6/7/2024,DN24159,Not Completed,Not Completed,Not Completed,Not Completed,Not Completed,Not Completed,Not Completed


In [72]:
#Bring in Older data and add to it
JoinedCSV = pd.read_csv(JoinedCSVpath)
#merge Data together
dataframes = [JoinedCSV,new_data_dbf]
updated_df = pd.concat(dataframes)
updated_df.reset_index(drop=True)
updated_df.shape
#make sure there aren't any duplicate ID's
updated_df.drop_duplicates(subset='SiteID',keep='last', inplace=True)
updated_df.head()

#Update CSV Version of points
updated_df.to_csv(JoinedCSVpath)

In [73]:
#Bring in the original feature layer collection

boem_flayer_collection = FeatureLayerCollection.fromitem(ssItem)
#boem_flayer_collection = FeatureLayerCollection.fromitem(ssITEM)
boem_flayer_collection.properties.layers[0].name

'BOEM_CLB_Sample_Sites'

In [74]:
#the Overwrite
boem_flayer_collection.manager.overwrite(JoinedCSVpath)

{'success': True}